## Few-shot Sentence Classification
This notebook trains a few-shot classifier to label parliamentary sentences as `opinionated` or `non_opinionated`.
It loads the dataset, builds a small stratified training set, trains a SetFit model, and evaluates performance on the remaining data.

In [2]:
import pandas as pd
import os
from pathlib import Path

import sys
sys.path.append('../src') # add src directory to path

from utils import format_discussion
from setfit import SetFitModel, SetFitTrainer
from sentence_transformers.losses import CosineSimilarityLoss

In [3]:
dataset = pd.read_csv('../data/opinion_dataset.csv')

In [4]:
dataset.shape

(60, 2)

In [5]:
#create test and train set (8 samples per class for training and the rest is testing)
train_data = dataset.groupby('target').apply(lambda x: x.sample(8, random_state=42)).reset_index(drop=True)
#test set is the rest of the data
test_data = dataset.drop(train_data.index).reset_index(drop=True)
test_data.shape, train_data.shape

/var/folders/06/25_1mnt97hq60zy5672y3qzw0000gn/T/ipykernel_63732/868968334.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_data = dataset.groupby('target').apply(lambda x: x.sample(8, random_state=42)).reset_index(drop=True)


((44, 2), (16, 2))

In [6]:
#converting to hugging face format
from datasets import Dataset

#rename target column to label (since its required by setfit)
train_data = train_data.rename(columns={'target': 'label'})
test_data = test_data.rename(columns={'target': 'label'})

train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

SetFit is used here as the few-shot approach for binary sentence classification (`opinionated` vs `non_opinionated`).

In [7]:
# Load SetFit model from Hub
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-mpnet-base-v2")

# Create trainer
trainer = SetFitTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    loss_class=CosineSimilarityLoss,
    batch_size=16,
    num_iterations=20, # Number of text pairs to generate for contrastive learning
    num_epochs=1 # Number of epochs to use for contrastive learning
)

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/var/folders/06/25_1mnt97hq60zy5672y3qzw0000gn/T/ipykernel_63732/2232557680.py:5: DeprecationWarning: `SetFitTrainer` has been deprecated and will be removed in v2.0.0 of SetFit. Please use `Trainer` instead.
  trainer = SetFitTrainer(


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

In [8]:
trainer.train()
metrics = trainer.evaluate()

***** Running training *****
  Num unique pairs = 640
  Batch size = 16
  Num epochs = 1
/Users/andreacristiano/Desktop/VNIVERSITA/MAGISTRALE 2 ANNO/Big Data analytics/stance-detection-eu-parliaments/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,0.235300


***** Running evaluation *****


In [9]:
metrics

{'accuracy': 0.9545454545454546}

In [10]:
model.save_pretrained( save_directory='./../models/setfit-fewshot-opinions-1')

In [ ]:
model.push_to_hub("andreacristiano/stancedetection")